# Notebook 4: Generate Kaiaulu Config Files

At this point, we should know the Apache project keys in the Jira Emotion dataset and the date range per project.

The next step is to actually download the comment data for those projects from Jira. This is where Kaiaulu comes in.

This notebook creates one Kaiaulu YAML config file for each Apache project that has emotion-labeled comments in the database. Kaiaulu downloads and parses Jira comment data using `.yml` config files. Each config tells Kaiaulu which project to target and where to save the output.

Each config only needs two things, the domain URL and the project key:

```yaml
issue_tracker:
  jira:
    project_key_1:
      domain: https://issues.apache.org/jira
      project_key: <PROJECT_KEY>
      issues: ../../rawdata/<PROJECT_KEY>/jira/issues/
      issue_comments: ../../rawdata/<PROJECT_KEY>/jira/issue_comments/
```

All Apache projects share the same Jira domain (`https://issues.apache.org/jira`), so only the `project_key` changes between files.

**Note:** Notebook 3 computed the date range of labeled comments per project. Use those bounds when downloading from Jira to avoid pulling unnecessary data and potentially getting an IP block.

### Prerequisites

Before running this notebook:

1. **Notebook 3 must have been run**: the Apache project keys and date ranges must already be in the database.
2. **PostgreSQL must be running locally** with the `jira` database loaded.
3. **`pyyaml` must be installed**: run `pip install pyyaml` in your virtual environment if needed.

### Step 1: Import dependencies

In [ ]:
import os
import yaml
import pandas as pd
from pathlib import Path
from sqlalchemy import create_engine, text

### Step 2: Set configuration

Update the variables below to match your local setup.`KAIAULU_REPO` is the path to your local `kaiaulu` repo root.

In [ ]:
# PostgreSQL connection details
PG_HOST     = "localhost"
PG_PORT     = 5432
PG_USER     = "postgres"
PG_PASSWORD = "ADD_YOUR_PASSWORD_HERE"
PG_DB       = "jira"

# Kaiaulu repo root
KAIAULU_REPO = Path(os.path.expanduser("~/PATH_TO/kaiaulu"))

# Derived paths
CONF_DIR = KAIAULU_REPO / "conf"

# Apache Jira domain (all ASF projects share this URL)
APACHE_JIRA_DOMAIN = "https://issues.apache.org/jira"

# Path written inside generated YAML files (relative to kaiaulu/vignettes/)
KAIAULU_RAWDATA_BASE = "../../rawdata/"

# Toggle writing config files to conf/
WRITE_CONFIGS = True

engine = create_engine(f"postgresql+psycopg2://{PG_USER}:{PG_PASSWORD}@{PG_HOST}:{PG_PORT}/{PG_DB}")
print("Connected to PostgreSQL.")
print(f"Config files will be written to: {CONF_DIR}")

### Step 3: Query Apache date ranges and project keys

The `download_jira_issues.Rmd` vignette allows you to download Jira issue comments by date range and project keys. This query pulls the Jira date ranges and project keys for all Apache projects with emotion-labeled comments. We can use the information from this query to know what Jira comments to download and from which date time frame.

In [ ]:
with engine.connect() as con:
    apache_projects = pd.read_sql(text("""
        SELECT
            SPLIT_PART(r.key, '-', 1)         AS project_key,
            r.project_name,
            COUNT(DISTINCT e.comment_id)       AS labeled_comment_count,
            MIN(c.creationdate)::date          AS date_lower_bound,
            MAX(c.creationdate)::date          AS date_upper_bound
        FROM jira_comment_emotion e
        INNER JOIN jira_issue_comment c ON e.comment_id = c.id
        INNER JOIN jira_issue_report r  ON c.issue_report_id = r.id
        WHERE r.repositoryname = 'ASF'
        GROUP BY project_key, r.project_name
        ORDER BY labeled_comment_count DESC;
    """), con)

print(f"Apache projects to generate configs for: {len(apache_projects)}")
display(apache_projects)

### Step 4: Query issue key range per project

`download_jira_issues.Rmd` requires the lowest and highest issue keys for each project that contain labeled comments. This query gives you those bounds.

In [ ]:
with engine.connect() as con:
    key_bounds = pd.read_sql(text("""
        SELECT
            SPLIT_PART(r.key, '-', 1)     AS project_key,
            r.project_name,
            COUNT(DISTINCT e.comment_id)  AS labeled_comment_count,
            MIN(CAST(SPLIT_PART(r.key, '-', 2) AS INTEGER)) AS key_lower_num,
            MAX(CAST(SPLIT_PART(r.key, '-', 2) AS INTEGER)) AS key_upper_num
        FROM jira_comment_emotion e
        INNER JOIN jira_issue_comment c ON e.comment_id = c.id
        INNER JOIN jira_issue_report r  ON c.issue_report_id = r.id
        WHERE r.repositoryname = 'ASF'
        GROUP BY project_key, r.project_name
        ORDER BY labeled_comment_count DESC;
    """), con)

key_bounds['issue_key_lower_bound'] = key_bounds['project_key'] + '-' + key_bounds['key_lower_num'].astype(str)
key_bounds['issue_key_upper_bound'] = key_bounds['project_key'] + '-' + key_bounds['key_upper_num'].astype(str)
key_bounds = key_bounds[['project_key', 'project_name', 'labeled_comment_count',
                          'issue_key_lower_bound', 'issue_key_upper_bound']]

print('Issue key bounds for Apache projects with labeled comments:')
print('Use these as issue_key_lower_bound and issue_key_upper_bound in download_jira_issues.Rmd')
display(key_bounds)


### Step 5: Generate and write config files

For each Apache project key in Step 3, we create a Kaiaulu YAML config with the Jira issue tracker.

Each file is named `project_key.yml` and saved to the `conf/` folder in Kaiaulu.

In [ ]:
KAIAULU_HEADER = (
    "# -*- yaml -*-\n"
    "# https://github.com/sailuh/kaiaulu\n"
    "#\n"
    "# Copying and distribution of this file, with or without modification,\n"
    "# are permitted in any medium without royalty provided the copyright\n"
    "# notice and this notice are preserved.  This file is offered as-is,\n"
    "# without any warranty.\n"
    "\n"
    "# Project Configuration File #\n"
    "#\n"
    "# To perform analysis on open source projects, you need to manually\n"
    "# collect some information from the project's website. As there is\n"
    "# no standardized website format, this file serves to distill\n"
    "# important data source information so it can be reused by others\n"
    "# and understood by Kaiaulu.\n"
    "#\n"
    "# Please check https://github.com/sailuh/kaiaulu/tree/master/conf to\n"
    "# see if a project configuration file already exists. Otherwise, we\n"
    "# would appreciate if you share your curated file with us by sending a\n"
    "# Pull Request: https://github.com/sailuh/kaiaulu/pulls\n"
    "#\n"
    "# Note, you do NOT need to specify this entire file to conduct analysis.\n"
    "# Each R Notebook uses a different portion of this file. To know what\n"
    "# information is used, see the project configuration file section at\n"
    "# the start of each R Notebook.\n"
    "#\n"
    "# Please comment unused parameters instead of deleting them for clarity.\n"
    "# If you have questions, please open a discussion:\n"
    "# https://github.com/sailuh/kaiaulu/discussions\n"
)

written = []
skipped = []

for _, row in apache_projects.iterrows():
    key        = row['project_key']
    key_lower  = key.lower()
    out_path   = CONF_DIR / f"{key_lower}.yml"

    if os.path.exists(out_path):
        skipped.append(key)
        continue

    config = {
        'project': {
            'website': f"https://issues.apache.org/jira/projects/{key}"
        },
        'issue_tracker': {
            'jira': {
                'project_key_1': {
                    'domain':         APACHE_JIRA_DOMAIN,
                    'project_key':    key,
                    'issues':         f"{KAIAULU_RAWDATA_BASE}jira/{key_lower}/issues/",
                    'issue_comments': f"{KAIAULU_RAWDATA_BASE}jira/{key_lower}/issue_comments/"
                }
            }
        }
    }

    if WRITE_CONFIGS:
        with open(out_path, 'w') as f:
            f.write(KAIAULU_HEADER)
            yaml.dump(config, f, default_flow_style=False, sort_keys=False)
    written.append(key)

print(f"Config files written: {len(written)}")
for k in written:
    print(f"  {k.lower()}.yml")

if skipped:
    print(f"\nSkipped (file already exists): {len(skipped)}")
    for k in skipped:
        print(f"  {k.lower()}.yml")

### Step 6: Verify the generated configs

Read back one of the generated files to confirm the structure looks correct before using it with Kaiaulu.

In [ ]:
if written:
    sample_key  = written[0].lower()
    sample_path = os.path.join(KAIAULU_REPO, f"{sample_key}.yml")
    with open(sample_path) as f:
        print(f"Contents of {sample_key}.yml:\n")
        print(f.read())
else:
    print("No new files were written. All configs were skipped (already exist).")

### What comes next

Before opening Notebook 5, download the Jira issue and comment data for each project key using the `download_jira_issues.Rmd` vignette in `kaiaulu`:

1. **Load the config**: Update the config call to point to your project's YAML file (e.g., `../conf/zookeeper.yml`).

2. **Create the rawdata directories**:
   ```
   mkdir -p rawdata/<project_key>/jira/issues/
   mkdir -p rawdata/<project_key>/jira/issue_comments/
   ```

3. **Download Issues by Date Range**: Use `download_jira_issues_by_date()` with the project date range.

4. **Download Issues by Key Range**: Use `download_jira_issues_by_issue_key()` with the project key range.

Once you've done the above steps, run Notebook 5 to join the emotion labels to the downloaded comment data.